# UNet Water Bodies — Sequential Experiments

Runs three experiments back-to-back in a single Colab session.
The scene split, normalisation statistics, and class weights are computed
once and shared across all experiments so comparisons are fair.

| # | Name | Bands | Norm | Training data |
|---|------|-------|------|---------------|
| 1 | `baseline_indices_only` | b7–b10 (NDVI, NDWI, NDRE, NISI) | global z-score | Mukherjee only |
| 2 | `optionB_bands_only` | b1–b6 (spectral) | per-source z-score | Mukherjee + local |
| 3 | `optionB_bands_and_indices` | b1–b10 (all) | per-source z-score | Mukherjee + local |

Experiment 1 was already run in `unet_water_bodies_colab.ipynb`. Re-running it here
ensures all three share the same random seed, scene split, and class weights.

---
## 0.1 Install Dependencies

In [ ]:
import subprocess, sys
pkgs = ["rasterio", "rioxarray", "geopandas", "albumentations",
        "torchinfo", "torchmetrics>=1.3", "einops"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)
print("Installation complete")

---
## 0.2 Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
print("Drive mounted")

---
## 0.3 Configuration

In [ ]:
from pathlib import Path

# ── Google Drive paths ────────────────────────────────────────────────────────
DRIVE_ROOT      = Path("/content/drive/MyDrive/Research/EO-Methane/paper2")
DATA_DIR        = DRIVE_ROOT / "data" / "Mukherjee_2024"
IMAGE_DIR       = DATA_DIR / "PS"        # 10-band preprocessed PlanetScope GeoTIFFs
MASK_DIR        = DATA_DIR / "labels"    # binary water mask GeoTIFFs

LOCAL_DATA_DIR  = DRIVE_ROOT / "data" / "local_nc"
LOCAL_IMAGE_DIR = LOCAL_DATA_DIR / "PS"
LOCAL_MASK_DIR  = LOCAL_DATA_DIR / "labels"
DEM_DIR         = None

# Shared checkpoint root — each experiment writes to its own subdirectory
EXPERIMENTS_DIR = DRIVE_ROOT / "experiments"
EXPERIMENTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Training (shared across all experiments) ──────────────────────────────────
CHIP_SIZE          = 256
BATCH_SIZE         = 16
NUM_EPOCHS         = 50
LEARNING_RATE      = 1e-4
N_VAL_SCENES       = 8
N_TEST_SCENES      = 8
PINNED_TEST_SCENES = ["SID78"]
RANDOM_SEED        = 42
ENCODER_CHANNELS   = [16, 32, 64, 128]

print(f"Experiments dir: {EXPERIMENTS_DIR}")

---
## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import pandas as pd
import random
import copy
import warnings
warnings.filterwarnings("ignore")

import rioxarray as rxr
import rasterio
from rasterio.windows import Window

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchinfo import summary
import torchmetrics

import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm.notebook import tqdm

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU — running on CPU")
print(f"PyTorch: {torch.__version__}")

---
## 2. Shared Setup
Runs once. Scene split, normalisation stats, and class weights are fixed
for all three experiments.

In [ ]:
# ── Helper functions ──────────────────────────────────────────────────────────

def preprocess_mask(raw_mask: np.ndarray) -> np.ndarray:
    """
    Convert 4-class Mukherjee labels to binary + ignore.
    0=nodata → 255, 1=not-water → 0, 2=low-conf → 255, 3=water → 1
    """
    out = np.full_like(raw_mask, 255, dtype=np.uint8)
    out[raw_mask == 1] = 0
    out[raw_mask == 3] = 1
    return out


def load_chip(path, row, col, chip_size, band_indices=None):
    window = Window(col, row, chip_size, chip_size)
    with rasterio.open(path) as src:
        data = src.read(
            [b + 1 for b in band_indices] if band_indices is not None else None,
            window=window
        )
    return data.astype(np.float32)


def list_paired_files(image_dir, mask_dir, source):
    pairs = []
    for img_path in sorted(image_dir.glob("*.tif")):
        mask_path = mask_dir / img_path.name
        if mask_path.exists():
            pairs.append((img_path, mask_path, source))
        else:
            print(f"  Warning: no mask for {img_path.name}")
    print(f"  [{source}] {len(pairs)} scenes")
    return pairs


def find_scene_index(pairs, scene_id):
    for i, (img_path, _, _) in enumerate(pairs):
        if scene_id.lower() in img_path.stem.lower():
            return i
    raise ValueError(f"Scene '{scene_id}' not found.")


# ── Load paired files ─────────────────────────────────────────────────────────
print("Loading paired files...")
mukherjee_pairs = list_paired_files(IMAGE_DIR,       MASK_DIR,       "mukherjee")
local_pairs     = list_paired_files(LOCAL_IMAGE_DIR, LOCAL_MASK_DIR, "local")
pairs           = mukherjee_pairs + local_pairs
print(f"Total: {len(pairs)} scenes")

# ── Chip index ────────────────────────────────────────────────────────────────
def build_chip_index(pairs, chip_size=256):
    chips = []
    for i, (img_path, _, _) in enumerate(pairs):
        with rasterio.open(img_path) as src:
            H, W = src.height, src.width
        for r in range(0, H - chip_size + 1, chip_size):
            for c in range(0, W - chip_size + 1, chip_size):
                chips.append((i, r, c))
    m = sum(1 for c in chips if pairs[c[0]][2] == "mukherjee")
    l = sum(1 for c in chips if pairs[c[0]][2] == "local")
    print(f"Total chips: {len(chips)}  (mukherjee: {m}, local: {l})")
    return chips

chip_index = build_chip_index(pairs, chip_size=CHIP_SIZE)

# ── Scene-level split (fixed for all experiments) ─────────────────────────────
pinned_test_indices = set()
for sid in PINNED_TEST_SCENES:
    idx = find_scene_index(pairs, sid)
    pinned_test_indices.add(idx)
    print(f"Pinned to test: {pairs[idx][0].name}  [{pairs[idx][2]}]")

remaining    = [i for i in range(len(pairs)) if i not in pinned_test_indices]
random.shuffle(remaining)
n_extra_test = N_TEST_SCENES - len(pinned_test_indices)

test_scene_idx  = pinned_test_indices | set(remaining[:n_extra_test])
val_scene_idx   = set(remaining[n_extra_test:n_extra_test + N_VAL_SCENES])
train_scene_idx = set(remaining[n_extra_test + N_VAL_SCENES:])

for split_name, split_idx in [("train", train_scene_idx),
                               ("val",   val_scene_idx),
                               ("test",  test_scene_idx)]:
    m = sum(1 for i in split_idx if pairs[i][2] == "mukherjee")
    l = sum(1 for i in split_idx if pairs[i][2] == "local")
    print(f"{split_name:5s}: {len(split_idx):3d} scenes  (mukherjee: {m}, local: {l})")

train_chips = [c for c in chip_index if c[0] in train_scene_idx]
val_chips   = [c for c in chip_index if c[0] in val_scene_idx]
test_chips  = [c for c in chip_index if c[0] in test_scene_idx]

In [ ]:
# ── Dataset class ─────────────────────────────────────────────────────────────
class WaterBodyDataset(Dataset):
    """
    _stats: dict {source: (mean, std)} for per-source normalisation,
            or {"global": (mean, std)} for global normalisation.
    When using global stats, all chips use the same mean/std regardless of source.
    """
    def __init__(self, pairs, chip_index, spectral_bands,
                 chip_size=256, transform=None):
        self.pairs          = pairs
        self.chip_index     = chip_index
        self.spectral_bands = spectral_bands
        self.chip_size      = chip_size
        self.transform      = transform
        self._stats         = {}

    def __len__(self):
        return len(self.chip_index)

    def _get_raw(self, idx):
        pair_i, row, col         = self.chip_index[idx]
        img_path, mask_path, source = self.pairs[pair_i]
        image = load_chip(img_path, row, col, self.chip_size, self.spectral_bands)
        mask  = load_chip(mask_path, row, col, self.chip_size).squeeze(0)
        return image, mask, source

    def compute_stats(self, mode="per_source", n_samples=200):
        """
        mode="per_source" : separate mean/std per source (for experiments 2 & 3)
        mode="global"     : single mean/std across all training chips (experiment 1)
        """
        if mode == "per_source":
            chips_by_source = {}
            for i, (pair_i, _, _) in enumerate(self.chip_index):
                src = self.pairs[pair_i][2]
                chips_by_source.setdefault(src, []).append(i)
            for source, chip_list in chips_by_source.items():
                print(f"  Computing stats for '{source}' ({len(chip_list)} chips)...")
                idx = np.random.choice(len(chip_list), min(n_samples, len(chip_list)), replace=False)
                data = np.concatenate(
                    [self._get_raw(chip_list[i])[0].reshape(len(self.spectral_bands), -1)
                     for i in tqdm(idx, desc=source, leave=False)], axis=1)
                mean = np.nanmean(data, axis=1)
                std  = np.nanstd(data,  axis=1) + 1e-6
                self._stats[source] = (mean, std)
                print(f"    mean: {np.round(mean, 3)}")
                print(f"    std:  {np.round(std,  3)}")
        else:  # global
            print(f"  Computing global stats ({len(self.chip_index)} chips available)...")
            idx  = np.random.choice(len(self), min(n_samples, len(self)), replace=False)
            data = np.concatenate(
                [self._get_raw(i)[0].reshape(len(self.spectral_bands), -1)
                 for i in tqdm(idx, desc="global", leave=False)], axis=1)
            mean = np.nanmean(data, axis=1)
            std  = np.nanstd(data,  axis=1) + 1e-6
            self._stats["global"] = (mean, std)
            print(f"    mean: {np.round(mean, 3)}")
            print(f"    std:  {np.round(std,  3)}")
        return self._stats

    def __getitem__(self, idx):
        image, mask, source = self._get_raw(idx)
        mask  = preprocess_mask(mask.astype(np.uint8))
        # Use per-source stats if available, fall back to global
        key   = source if source in self._stats else "global"
        if key in self._stats:
            mean, std = self._stats[key]
            image = (image - mean[:, None, None]) / std[:, None, None]
        image = np.nan_to_num(image, nan=0.0, posinf=1.0, neginf=-1.0)
        if self.transform:
            aug   = self.transform(image=image.transpose(1, 2, 0),
                                   mask=mask.astype(np.float32))
            image = aug["image"]
            mask  = aug["mask"].long()
        else:
            image = torch.from_numpy(image)
            mask  = torch.from_numpy(mask.astype(np.int64))
        return image, mask


# ── Augmentation ──────────────────────────────────────────────────────────────
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.4),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.4),
    A.GaussNoise(var_limit=(0.001, 0.005), p=0.3),
    ToTensorV2()
])
val_transform = A.Compose([ToTensorV2()])

print("Shared setup complete.")

In [ ]:
# ── Model, loss, training utilities ──────────────────────────────────────────

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)


class UNet(nn.Module):
    def __init__(self, in_channels, out_channels=2, encoder_chn=None):
        super().__init__()
        if encoder_chn is None:
            encoder_chn = [16, 32, 64, 128]
        self.encoders = nn.ModuleList()
        self.pools    = nn.ModuleList()
        prev = in_channels
        for ch in encoder_chn:
            self.encoders.append(DoubleConv(prev, ch))
            self.pools.append(nn.MaxPool2d(2))
            prev = ch
        self.bottleneck = DoubleConv(prev, prev * 2)
        prev = prev * 2
        self.upconvs  = nn.ModuleList()
        self.decoders = nn.ModuleList()
        for ch in reversed(encoder_chn):
            self.upconvs.append(nn.ConvTranspose2d(prev, ch, 2, stride=2))
            self.decoders.append(DoubleConv(ch * 2, ch))
            prev = ch
        self.output_conv = nn.Conv2d(prev, out_channels, 1)

    def forward(self, x):
        skips = []
        for enc, pool in zip(self.encoders, self.pools):
            x = enc(x); skips.append(x); x = pool(x)
        x = self.bottleneck(x)
        for up, dec, skip in zip(self.upconvs, self.decoders, reversed(skips)):
            x = up(x)
            if x.shape != skip.shape:
                x = F.pad(x, [0, skip.shape[3]-x.shape[3], 0, skip.shape[2]-x.shape[2]])
            x = dec(torch.cat([skip, x], dim=1))
        return self.output_conv(x)


class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0): super().__init__(); self.smooth = smooth
    def forward(self, logits, targets):
        valid     = targets != 255
        probs     = torch.softmax(logits, dim=1)[:, 1][valid]
        targets_f = targets[valid].float()
        intersect = (probs * targets_f).sum()
        return 1.0 - (2.*intersect + self.smooth) / (probs.sum() + targets_f.sum() + self.smooth)


class CombinedLoss(nn.Module):
    def __init__(self, class_weights):
        super().__init__()
        self.ce   = nn.CrossEntropyLoss(weight=class_weights, ignore_index=255)
        self.dice = DiceLoss()
    def forward(self, logits, targets):
        return 0.5 * self.ce(logits, targets) + 0.5 * self.dice(logits, targets)


def make_metrics():
    return {
        "iou":       torchmetrics.JaccardIndex(task="binary", ignore_index=255).to(DEVICE),
        "f1":        torchmetrics.F1Score(task="binary",      ignore_index=255).to(DEVICE),
        "precision": torchmetrics.Precision(task="binary",    ignore_index=255).to(DEVICE),
        "recall":    torchmetrics.Recall(task="binary",       ignore_index=255).to(DEVICE),
    }


def run_epoch(loader, model, criterion, metrics, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss = 0.0
    for m in metrics.values(): m.reset()
    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for images, masks in loader:
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            logits = model(images)
            loss   = criterion(logits, masks)
            if is_train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item()
            preds = logits.argmax(dim=1)
            for m in metrics.values(): m.update(preds, masks)
    return {
        "loss": total_loss / len(loader),
        "iou":  metrics["iou"].compute().item(),
        "f1":   metrics["f1"].compute().item(),
    }


print("Model and training utilities defined.")

---
## 3. `run_experiment()` — Core Training Function

Accepts an experiment config dict and runs the full train → evaluate → log pipeline.
Returns a results dict for the comparison plot.

In [ ]:
def run_experiment(cfg):
    """
    cfg keys:
        name          : str   experiment name used for logging and checkpoint dir
        spectral_bands: list  0-based band indices into the 10-band stack
        norm_mode     : str   "global" or "per_source"
        training_data : str   "mukherjee" or "mukherjee+local" (for log only)
        notes         : str   free-text note for the experiment log
    """
    print(f"\n{'='*60}")
    print(f"  Experiment: {cfg['name']}")
    print(f"  Bands:      {cfg['spectral_bands']}")
    print(f"  Norm:       {cfg['norm_mode']}")
    print(f"{'='*60}\n")

    n_channels  = len(cfg["spectral_bands"])
    ckpt_dir    = EXPERIMENTS_DIR / cfg["name"]
    ckpt_dir.mkdir(exist_ok=True)
    best_path   = ckpt_dir / "best_model.pt"
    resume_path = ckpt_dir / "latest_checkpoint.pt"
    stats_path  = ckpt_dir / "norm_stats.npy"

    # ── Build datasets with this experiment's band selection ─────────────────
    train_ds = WaterBodyDataset(pairs, train_chips, cfg["spectral_bands"],
                                chip_size=CHIP_SIZE, transform=train_transform)
    val_ds   = WaterBodyDataset(pairs, val_chips,   cfg["spectral_bands"],
                                chip_size=CHIP_SIZE, transform=val_transform)
    test_ds  = WaterBodyDataset(pairs, test_chips,  cfg["spectral_bands"],
                                chip_size=CHIP_SIZE, transform=val_transform)

    # ── Normalisation stats ───────────────────────────────────────────────────
    if stats_path.exists():
        print("Loading saved normalisation stats...")
        stats = np.load(stats_path, allow_pickle=True).item()
    else:
        print("Computing normalisation stats...")
        stats = train_ds.compute_stats(mode=cfg["norm_mode"])
        np.save(stats_path, stats)
    train_ds._stats = stats
    val_ds._stats   = stats
    test_ds._stats  = stats

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=2, pin_memory=True)

    # ── Class weights (computed fresh per experiment — band selection affects
    #    which chips are sampled but not the mask values, so this is stable) ──
    water_px, total_px = 0, 0
    for i in np.random.choice(len(train_ds), min(100, len(train_ds)), replace=False):
        _, mask, _ = train_ds._get_raw(i)
        mask = preprocess_mask(mask.astype(np.uint8))
        valid = mask != 255
        water_px += (mask[valid] == 1).sum()
        total_px += valid.sum()
    water_frac = water_px / total_px
    w_w = 1.0 / (water_frac + 1e-6)
    w_n = 1.0 / (1.0 - water_frac + 1e-6)
    class_weights = torch.tensor([w_n/(w_w+w_n)*2, w_w/(w_w+w_n)*2],
                                  dtype=torch.float32).to(DEVICE)
    print(f"Water fraction: {water_frac*100:.2f}%  "
          f"weights — non-water: {class_weights[0]:.3f}, water: {class_weights[1]:.3f}")
    criterion = CombinedLoss(class_weights)

    # ── Model (fresh weights every experiment) ────────────────────────────────
    torch.manual_seed(RANDOM_SEED)   # same initialisation across experiments
    model = UNet(in_channels=n_channels, encoder_chn=ENCODER_CHANNELS).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=5)
    metrics   = make_metrics()

    # ── Resume if checkpoint exists ───────────────────────────────────────────
    start_epoch   = 1
    best_val_loss = float("inf")
    history       = {"train_loss": [], "val_loss": [],
                     "train_iou":  [], "val_iou":  [],
                     "train_f1":   [], "val_f1":   []}

    if resume_path.exists():
        ckpt = torch.load(resume_path, map_location=DEVICE, weights_only=True)
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["scheduler"])
        start_epoch   = ckpt["epoch"] + 1
        best_val_loss = ckpt["best_val_loss"]
        history       = ckpt["history"]
        print(f"Resumed from epoch {ckpt['epoch']}")

    # ── Training loop ─────────────────────────────────────────────────────────
    for epoch in range(start_epoch, NUM_EPOCHS + 1):
        tr = run_epoch(train_loader, model, criterion, metrics, optimizer)
        vl = run_epoch(val_loader,   model, criterion, metrics)
        scheduler.step(vl["loss"])
        lr = optimizer.param_groups[0]["lr"]

        for k in ["loss", "iou", "f1"]:
            history[f"train_{k}"].append(tr[k])
            history[f"val_{k}"].append(vl[k])

        if vl["loss"] < best_val_loss:
            best_val_loss = vl["loss"]
            torch.save(model.state_dict(), best_path)

        torch.save({"epoch": epoch, "model": model.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "scheduler": scheduler.state_dict(),
                    "best_val_loss": best_val_loss,
                    "history": history}, resume_path)

        print(f"[{cfg['name']}] Epoch {epoch:03d}/{NUM_EPOCHS} "
              f"| train loss {tr['loss']:.4f}  IoU {tr['iou']:.3f}  F1 {tr['f1']:.3f} "
              f"| val loss {vl['loss']:.4f}  IoU {vl['iou']:.3f}  F1 {vl['f1']:.3f} "
              f"| lr {lr:.2e}")

    # ── Test evaluation ───────────────────────────────────────────────────────
    model.load_state_dict(torch.load(best_path, map_location=DEVICE, weights_only=True))
    test_metrics = make_metrics()
    model.eval()
    test_loss = 0.0
    with torch.no_grad():
        for images, masks in tqdm(test_loader, desc="Test", leave=False):
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            logits = model(images)
            test_loss += criterion(logits, masks).item()
            preds = logits.argmax(dim=1)
            for m in test_metrics.values(): m.update(preds, masks)

    results = {
        "loss":      test_loss / len(test_loader),
        "iou":       test_metrics["iou"].compute().item(),
        "f1":        test_metrics["f1"].compute().item(),
        "precision": test_metrics["precision"].compute().item(),
        "recall":    test_metrics["recall"].compute().item(),
        "history":   history,
    }

    print(f"\n── {cfg['name']} Test Results ──────────────")
    for k, v in results.items():
        if k != "history":
            print(f"  {k:10s}: {v:.4f}")

    # ── Training curves ───────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    epochs = range(1, len(history["train_loss"]) + 1)
    for ax, metric in zip(axes, ["loss", "iou", "f1"]):
        ax.plot(epochs, history[f"train_{metric}"], label="Train")
        ax.plot(epochs, history[f"val_{metric}"],   label="Val", linestyle="--")
        ax.set_title({"loss":"Loss","iou":"IoU","f1":"F1"}[metric])
        ax.set_xlabel("Epoch"); ax.legend()
    plt.suptitle(cfg["name"], fontsize=12)
    plt.tight_layout()
    plt.savefig(ckpt_dir / "training_curves.png", dpi=150)
    plt.show()

    # ── Log to shared CSV ─────────────────────────────────────────────────────
    log_path = EXPERIMENTS_DIR / "experiment_log.csv"
    row = {
        "name":           cfg["name"],
        "spectral_bands": str(cfg["spectral_bands"]),
        "n_channels":     n_channels,
        "norm_mode":      cfg["norm_mode"],
        "training_data":  cfg["training_data"],
        "test_loss":      round(results["loss"],      4),
        "test_iou":       round(results["iou"],       4),
        "test_f1":        round(results["f1"],        4),
        "test_precision": round(results["precision"], 4),
        "test_recall":    round(results["recall"],    4),
        "notes":          cfg.get("notes", ""),
    }
    df = pd.concat([pd.read_csv(log_path), pd.DataFrame([row])], ignore_index=True) \
         if log_path.exists() else pd.DataFrame([row])
    df.to_csv(log_path, index=False)
    print(f"Logged to {log_path}")

    # Return results and model for optional further use
    results["model"]   = model
    results["stats"]   = stats
    results["test_ds"] = test_ds
    return results


print("run_experiment() defined — ready to run experiments.")

---
## 3.5 Copy Existing Exp 1 Checkpoint

Copies the best model and checkpoint from the baseline notebook into the
experiments directory so `run_experiment()` picks it up and skips retraining.
**Run this cell once, then it is safe to re-run (it checks before copying).**
If the source checkpoint does not exist, Exp 1 will train from scratch.

In [ ]:
import shutil

# ── Source: baseline notebook checkpoint directory ────────────────────────────
src_dir = DRIVE_ROOT / "checkpoints"

# ── Destination: experiments subdirectory for Exp 1 ─────────────────────────
dst_dir = EXPERIMENTS_DIR / "baseline_indices_only"
dst_dir.mkdir(parents=True, exist_ok=True)

files_to_copy = {
    "latest_checkpoint.pt": "latest_checkpoint.pt",
    "best_model.pt":        "best_model.pt",
    "norm_stats.npy":       "norm_stats.npy",   # may not exist if saved before fix
}

for src_name, dst_name in files_to_copy.items():
    src_path = src_dir / src_name
    dst_path = dst_dir / dst_name
    if src_path.exists():
        if dst_path.exists():
            print(f"  Already exists, skipping: {dst_name}")
        else:
            shutil.copy(src_path, dst_path)
            print(f"  Copied: {src_name} → {dst_dir.name}/{dst_name}")
    else:
        print(f"  Not found, skipping: {src_name}")

# Verify what's now in the destination
print(f"\nContents of {dst_dir}:")
for f in sorted(dst_dir.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:30s} {size_kb:8.1f} KB")

# Check whether the checkpoint has history (needed for overlay curves in Sec 5)
import torch
ckpt_path = dst_dir / "latest_checkpoint.pt"
if ckpt_path.exists():
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=True)
    n_epochs = len(ckpt.get("history", {}).get("val_loss", []))
    print(f"\nCheckpoint epoch: {ckpt.get('epoch', '?')}  "
          f"history length: {n_epochs} epochs")
    if n_epochs == 0:
        print("  Warning: history is empty — val curves overlay in Sec 5 will "
              "show a flat line for Exp 1. This is cosmetic only.")

---
## 4. Run Experiments

Each experiment is fully self-contained — fresh model weights (same random seed),
same scene split, same augmentation. Only bands and normalisation mode differ.

In [ ]:
# ── Experiment 1: Indices only, global normalisation ──────────────────────────
# Reproduces the baseline already run in unet_water_bodies_colab.ipynb.
# Included here so all three share the same session/split for a fair comparison.
results_1 = run_experiment({
    "name":           "baseline_indices_only",
    "spectral_bands": [6, 7, 8, 9],      # NDVI, NDWI, NDRE, NISI
    "norm_mode":      "global",
    "training_data":  "mukherjee",
    "notes":          "Indices only, global z-score, Mukherjee only, 50 epochs"
})

In [ ]:
# ── Experiment 2: Spectral bands only, per-source normalisation ───────────────
results_2 = run_experiment({
    "name":           "optionB_bands_only",
    "spectral_bands": [0, 1, 2, 3, 4, 5],  # coastal_blue, blue, green, red, rededge, nir
    "norm_mode":      "per_source",
    "training_data":  "mukherjee+local",
    "notes":          "Spectral bands only, per-source z-score, Mukherjee+local, 50 epochs"
})

In [ ]:
# ── Experiment 3: All bands + indices, per-source normalisation ───────────────
results_3 = run_experiment({
    "name":           "optionB_bands_and_indices",
    "spectral_bands": list(range(10)),     # all 10 bands
    "norm_mode":      "per_source",
    "training_data":  "mukherjee+local",
    "notes":          "All 10 bands, per-source z-score, Mukherjee+local, 50 epochs"
})

---
## 5. Results Comparison

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
log_path = EXPERIMENTS_DIR / "experiment_log.csv"
df = pd.read_csv(log_path)

# Show only the three current experiments in order
exp_order = ["baseline_indices_only", "optionB_bands_only", "optionB_bands_and_indices"]
df_show   = df[df["name"].isin(exp_order)].set_index("name").loc[exp_order]
display(df_show[["test_iou", "test_f1", "test_precision", "test_recall",
                 "norm_mode", "training_data"]].round(4))

In [ ]:
# ── Comparison bar chart ──────────────────────────────────────────────────────
metrics    = ["test_iou", "test_f1", "test_precision", "test_recall"]
labels     = ["IoU", "F1", "Precision", "Recall"]
exp_labels = ["1: Indices\n(global norm)",
              "2: Bands\n(per-source)",
              "3: Bands+Indices\n(per-source)"]
colours    = ["#4C72B0", "#55A868", "#C44E52"]

x     = np.arange(len(metrics))
width = 0.25

fig, ax = plt.subplots(figsize=(11, 5))
for i, (exp_name, label, colour) in enumerate(
        zip(exp_order, exp_labels, colours)):
    row    = df_show.loc[exp_name]
    values = [row[m] for m in metrics]
    bars   = ax.bar(x + i * width, values, width, label=label, color=colour, alpha=0.85)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{val:.3f}", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x + width)
ax.set_xticklabels(labels)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
ax.set_title("Test Metrics by Experiment")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(EXPERIMENTS_DIR / "experiment_comparison.png", dpi=150)
plt.show()

In [ ]:
# ── Training curve overlay — val IoU across all 3 experiments ────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for results, label, colour in [
    (results_1, "1: Indices (global)",        "#4C72B0"),
    (results_2, "2: Bands (per-source)",       "#55A868"),
    (results_3, "3: Bands+Indices (per-src)",  "#C44E52"),
]:
    h = results["history"]
    ep = range(1, len(h["val_loss"]) + 1)
    axes[0].plot(ep, h["val_loss"], label=label, color=colour)
    axes[1].plot(ep, h["val_iou"],  label=label, color=colour)

axes[0].set_title("Val Loss");     axes[0].set_xlabel("Epoch"); axes[0].legend(fontsize=8)
axes[1].set_title("Val IoU");      axes[1].set_xlabel("Epoch"); axes[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig(EXPERIMENTS_DIR / "val_curves_overlay.png", dpi=150)
plt.show()